# 0. Load in the cleaned test and train data

In [44]:
import pandas as pd

test = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv")
test.info()
submission = test[["GuestID"]].copy()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   GuestID         1739 non-null   int64  
 1   BookingDate     1739 non-null   object 
 2   PromoCode       1739 non-null   object 
 3   Region          1739 non-null   object 
 4   AllInclusive    1739 non-null   float64
 5   PackageType     1739 non-null   object 
 6   VIP             1739 non-null   float64
 7   RoomService     1739 non-null   float64
 8   Dining          1739 non-null   float64
 9   Retail          1739 non-null   float64
 10  Spa             1739 non-null   float64
 11  Entertainment   1739 non-null   float64
 12  LoyaltyPoints   1739 non-null   int64  
 13  SurveyScore     1739 non-null   int64  
 14  DaysSinceEmail  1739 non-null   int64  
 15  BookingChannel  1739 non-null   object 
 16  AgeGroup        1739 non-null   object 
 17  ReferralSource  1739 non-null   o

In [45]:
test.isna().sum()

GuestID           0
BookingDate       0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
SharedRoom        0
dtype: int64

In [46]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   GuestID         6954 non-null   int64  
 1   BookingDate     6954 non-null   object 
 2   PromoCode       6954 non-null   object 
 3   Region          6954 non-null   object 
 4   AllInclusive    6954 non-null   float64
 5   PackageType     6954 non-null   object 
 6   VIP             6954 non-null   float64
 7   RoomService     6954 non-null   float64
 8   Dining          6954 non-null   float64
 9   Retail          6954 non-null   float64
 10  Spa             6954 non-null   float64
 11  Entertainment   6954 non-null   float64
 12  LoyaltyPoints   6954 non-null   int64  
 13  SurveyScore     6954 non-null   int64  
 14  DaysSinceEmail  6954 non-null   int64  
 15  BookingChannel  6954 non-null   object 
 16  AgeGroup        6954 non-null   object 
 17  ReferralSource  6954 non-null   o

In [47]:
train.isna().sum()

GuestID           0
BookingDate       0
PromoCode         0
Region            0
AllInclusive      0
PackageType       0
VIP               0
RoomService       0
Dining            0
Retail            0
Spa               0
Entertainment     0
LoyaltyPoints     0
SurveyScore       0
DaysSinceEmail    0
BookingChannel    0
AgeGroup          0
ReferralSource    0
Churned           0
SharedRoom        0
dtype: int64

In [48]:
#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Train the model - This is where your model goes

In [49]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

# CatBoost handles categoricals natively — just tell it which columns are categorical.
# GuestID is an identifier, drop it from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

cat_features = [c for c in X.columns if X[c].dtype == "object"]
# CatBoost requires categorical columns to have no NaN — fill with a sentinel string.
X[cat_features] = X[cat_features].fillna("missing")

from sklearn.model_selection import train_test_split
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train_cb == 0).sum(), (y_train_cb == 1).sum()

cat = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    scale_pos_weight=neg / pos,
    eval_metric="AUC",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
)

train_pool = Pool(X_train_cb, y_train_cb, cat_features=cat_features)
test_pool = Pool(X_test_cb, y_test_cb, cat_features=cat_features)

cat.fit(train_pool, eval_set=test_pool)

probs = cat.predict_proba(X_test_cb)[:, 1]
print(f"Test ROC-AUC:        {roc_auc_score(y_test_cb, probs):.3f}")
print(f"Best iteration:      {cat.get_best_iteration()}")

# Feature importance
import pandas as pd
fi = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

Test ROC-AUC:        0.908
Best iteration:      220
       feature  importance
           Spa   13.512222
  AllInclusive   12.146245
 Entertainment   11.636843
        Region   11.330726
   RoomService    9.499333
   BookingDate    8.925455
ReferralSource    8.919083
     PromoCode    6.820105
        Dining    5.615530
        Retail    3.096350
      AgeGroup    2.673115
   PackageType    1.541503
   SurveyScore    1.367693
 LoyaltyPoints    1.218950
DaysSinceEmail    0.982398
BookingChannel    0.527062
           VIP    0.121577
    SharedRoom    0.065810


# 2. Test - This will produce the test data

In [50]:
#run the test data against the model to get the probabilities for the submission file

#drop GueestID Column from the test data
test = test.drop(columns=["GuestID"])
probs = cat.predict(test)

#probs = xgb.predict(test)

#display the probs
print(probs)

#remove all columns except for the GuestID column and then add the probabilities as a new column called "Churned"
#submission = train[["GuestID"]].copy()
submission["Churned"] = probs

submission.info()

#output the csv file with the predictions
submission.to_csv("submission.csv", index=False)

[1 0 0 ... 1 0 0]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   GuestID  1739 non-null   int64
 1   Churned  1739 non-null   int64
dtypes: int64(2)
memory usage: 27.3 KB
